# Capítulo 2 — Distribuições de Dados e Amostras

**Livro:** começa na página **71** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Viés de Seleção · PDF p. 79

#### 🧪 Exercício: Viés de sobrevivência

Simule 1.000 fundos com retorno anual normal(0, 10%). Fique só com os "sobreviventes" (retorno > −5%) e compare a média dos sobreviventes com a de todos.

In [ ]:
ret = rng.normal(0, 0.10, 1000)
print(f"todos {ret.mean():.3%} | sobreviventes {ret[ret > -0.05].mean():.3%}")

## Regressão à Média · PDF p. 81

#### 🧪 Exercício: Regressão à média

Cada jogador tem uma habilidade fixa e, em cada temporada, desempenho = habilidade + sorte. Simule 2 temporadas para 1.000 jogadores, pegue os 10% melhores da 1ª e veja a média deles na 2ª.

In [ ]:
hab = rng.normal(0, 1, 1000)
t1 = hab + rng.normal(0, 1, 1000); t2 = hab + rng.normal(0, 1, 1000)
top = t1 >= np.quantile(t1, 0.9)
print(round(t1[top].mean(), 2), round(t2[top].mean(), 2))  # caem na 2ª temporada

## Teorema de Limite Central · PDF p. 86

#### 🧪 Exercício: Distribuição amostral da média

Partindo de uma exponencial (bem assimétrica), tire 2.000 amostras de tamanho 5, 20 e 100 e desenhe o histograma da média amostral de cada caso.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, n in zip(ax, [5, 20, 100]):
    a.hist(rng.exponential(1, (2000, n)).mean(axis=1), bins=40); a.set_title(f"n = {n}")
plt.show()

## Erro-padrão · PDF p. 87

#### 🧪 Exercício: Erro-padrão: fórmula × simulação

Para amostras de tamanho 50 de uma normal(10, 3), compare o erro-padrão da média pela fórmula s/√n com o desvio-padrão de 5.000 médias simuladas.

In [ ]:
amostra = rng.normal(10, 3, 50)
print(round(amostra.std(ddof=1) / np.sqrt(50), 3), round(rng.normal(10, 3, (5000, 50)).mean(axis=1).std(), 3))

## O Bootstrap · PDF p. 88

#### 🧪 Exercício: Bootstrap da mediana

Com 200 rendas lognormais, estime o erro-padrão e o viés da mediana por *bootstrap* (2.000 reamostragens).

In [ ]:
renda = rng.lognormal(8, 1, 200)
boot = np.array([np.median(rng.choice(renda, renda.size)) for _ in range(2000)])
print(f"mediana {np.median(renda):.0f} | viés {boot.mean() - np.median(renda):.1f} | erro-padrão {boot.std(ddof=1):.1f}")

## Intervalos de Confiança · PDF p. 93

#### 🧪 Exercício: Intervalo bootstrap × intervalo t

Para 40 valores normais(100, 15), calcule o intervalo de 95% da média pelo bootstrap (percentis 2,5 e 97,5) e pela fórmula t (`stats.t.interval`).

In [ ]:
x = rng.normal(100, 15, 40)
boot = [rng.choice(x, x.size).mean() for _ in range(5000)]
print(np.percentile(boot, [2.5, 97.5]).round(2), np.round(stats.t.interval(0.95, x.size - 1, x.mean(), stats.sem(x)), 2))

## Normal Padrão e Gráficos QQ · PDF p. 98

#### 🧪 Exercício: Gráfico QQ

Faça gráficos QQ (`stats.probplot(..., plot=ax)`) para 300 valores normais e 300 valores lognormais. Como a assimetria aparece no gráfico?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(rng.normal(size=300), plot=ax[0]); ax[0].set_title("normal")
stats.probplot(rng.lognormal(size=300), plot=ax[1]); ax[1].set_title("lognormal"); plt.show()

## Distribuições de Cauda Longa · PDF p. 100

#### 🧪 Exercício: Cauda longa: eventos extremos

Compare a probabilidade de um valor passar de 4 desvios-padrão numa normal e numa t de Student com 3 graus de liberdade (padronizada pela sua variância). Quantas vezes é maior?

In [ ]:
p_norm = 2 * stats.norm.sf(4)
dp_t = np.sqrt(3 / (3 - 2))
p_t = 2 * stats.t.sf(4 * dp_t, 3)
print(f"{p_norm:.2e} {p_t:.2e} -> {p_t / p_norm:.0f}x")

## Distribuição t de Student · PDF p. 103

#### 🧪 Exercício: t × normal

Desenhe a densidade da normal padrão e da t com 2, 5 e 30 graus de liberdade. Para qual g.l. elas ficam quase iguais?

In [ ]:
xs = np.linspace(-4, 4, 300)
plt.plot(xs, stats.norm.pdf(xs), "k", lw=2, label="normal")
for gl in [2, 5, 30]: plt.plot(xs, stats.t.pdf(xs, gl), label=f"t({gl})")
plt.legend(); plt.show()

## Distribuição Binomial · PDF p. 106

#### 🧪 Exercício: Binomial: cliques em anúncios

Um anúncio tem 2% de cliques. Em 200 exibições, qual é a chance de 0 cliques, de exatamente 4 e de 8 ou mais? Confira a média e a variância por simulação.

In [ ]:
b = stats.binom(200, 0.02)
print(round(b.pmf(0), 4), round(b.pmf(4), 4), round(b.sf(7), 4))
sim = rng.binomial(200, 0.02, 100_000); print(sim.mean(), round(sim.var(), 3), b.mean(), b.var())

## Distribuições Poisson · PDF p. 109

#### 🧪 Exercício: Poisson: chamadas por minuto

Uma central recebe em média 2 chamadas por minuto. Qual é a probabilidade de 5 ou mais chamadas num minuto? Quantas chamadas cobrem 99% dos minutos?

In [ ]:
p = stats.poisson(2); print(round(p.sf(4), 4), p.ppf(0.99))

## Distribuição Exponencial · PDF p. 110

#### 🧪 Exercício: Tempo entre chamadas

Com a mesma taxa (2 por minuto), simule 10.000 tempos entre chamadas (exponencial) e estime P(espera > 1 min). Compare com e⁻².

In [ ]:
t = rng.exponential(1 / 2, 10_000); print(round((t > 1).mean(), 4), round(np.exp(-2), 4))

## Distribuição Weibull · PDF p. 112

#### 🧪 Exercício: Weibull e taxa de falha

Simule 5.000 tempos de vida de peças com Weibull de forma 1,5 e escala 1.000 h e ajuste com `stats.weibull_min.fit(..., floc=0)`. Os parâmetros voltam?

In [ ]:
vida = stats.weibull_min.rvs(1.5, scale=1000, size=5000, random_state=1)
forma, loc, escala = stats.weibull_min.fit(vida, floc=0); print(round(forma, 3), round(escala, 1))